In [2]:
import sys
sys.path.append("..")

import torch
from src.inference import load_model

model = load_model()
dummy = torch.randn(1, 4, 512, 512)   # fixed 512x512 input, the dataset's tile size

torch.onnx.export(
    model, dummy, "../models/forest_unet.onnx",
    input_names=["input"], output_names=["logits"],
    opset_version=17, dynamo=False,
)
print("exported")

/tmp/ipykernel_5248/4009669977.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


exported


In [3]:
import os
import numpy as np
import onnxruntime as ort
from src.dataset import ForestDataset

sess = ort.InferenceSession("../models/forest_unet.onnx", providers=["CPUExecutionProvider"])

x, _ = ForestDataset("atlantic", "Test")[0]
x = x[None]                                    # (1, 4, 512, 512), already preprocessed

with torch.no_grad():
    a = model(x).numpy()
b = sess.run(None, {"input": x.numpy()})[0]

print("max abs difference:", np.abs(a - b).max())
print("pixels with the same forest/non-forest decision:", ((a > 0) == (b > 0)).mean())
print("file size:", round(os.path.getsize("../models/forest_unet.onnx") / 1e6, 1), "MB")

max abs difference: 1.1444092e-05
pixels with the same forest/non-forest decision: 1.0
file size: 97.8 MB
